# Imports

In [5]:
import os
import random
import shutil
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from PIL import Image
import pillow_heif
import joblib
from scipy.stats import skew
import os
from collections import defaultdict

import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torchvision.transforms.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.datasets import ImageFolder

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split


import os
from collections import defaultdict
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset

from sklearn.model_selection import StratifiedShuffleSplit
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torchvision.transforms.v2 as v2
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import matplotlib.pyplot as plt
# Enable HEIF/HEIC support
pillow_heif.register_heif_opener()

# **1. Data Splitting**
- **Implement a pipeline that preserves visit date information and prevents temporal leakage.**

| Class                                   | Train_dates                                                                 | Validation_dates     | Train_num_images | Val_num_images | Train_percentage | Val_percentage |
|-----------------------------------------|----------------------------------------------------------------------------|----------------------|------------------|----------------|------------------|----------------|
| Capparis spinosa L.                     | 2025-01-31 ,2024-11-22 ,2023-12-22 ,2024-12-23 ,2025-02-21                 | 2025-01-04           | 515              | 142            | 0.783866058      | 0.216133942    |
| Diplotaxis harra (Forssk.) Boiss.       | 2024-11-22 ,2024-03-08 ,2025-01-04                                         | 2025-01-31           | 415              | 136            | 0.753176044      | 0.246823956    |
| Iphiona mucronata (Forssk.) Asch. & Schweinf. | 2025-01-31 (32) ,2024-12-23                                                 | 2025-01-31 (28)      | 111              | 28             | 0.798561151      | 0.201438849    |
| Ochradenus baccatus Delile              | 2024-11-22 ,2024-12-23 ,2025-01-04 ,2023-12-22 ,2025-01-31 ,2024-01-20     | 2024-03-08           | 1396             | 498            | 0.737064414      | 0.262935586    |
| Peganum harmala L.                      | 2025-01-04 (85)                                                            | 2025-04-01 (22)      | 85               | 22             | 0.794392523      | 0.205607477    |
| Tamarix nilotica (Ehrenb.) Bunge        | 2024-11-22 ,2024-12-23 ,2023-12-22 ,2025-02-21 ,2025-01-31 ,2024-03-08     | 2024-01-20 ,2025-01-04 | 1074             | 320            | 0.770444763      | 0.229555237    |
| **Total**                               |                                                                            |                      | **3596**         | **1146**       | **0.758329819**  | **0.241670181** |

## Single-Date Random Split

In [18]:
def split_images(src_dir, dest_dir, num_images, move=False, seed=42):
    """
    Move or copy a specific number of images from src_dir to dest_dir, 
    ignoring hidden files (e.g., .DS_Store).
    """
    # Make sure destination directory exists
    os.makedirs(dest_dir, exist_ok=True)

    # List all visible files in source (ignore hidden ones)
    all_files = [
        f for f in os.listdir(src_dir) 
        if os.path.isfile(os.path.join(src_dir, f)) and not f.startswith('.')
    ]

    if len(all_files) == 0:
        raise ValueError(f"No valid (non-hidden) files found in {src_dir}")

    if num_images > len(all_files):
        raise ValueError(f"Requested {num_images} images, but only {len(all_files)} available in {src_dir}")

    # Reproducible random sample
    random.seed(seed)
    selected_files = random.sample(all_files, num_images)

    # Copy or move
    for file_name in selected_files:
        src_path = os.path.join(src_dir, file_name)
        dest_path = os.path.join(dest_dir, file_name)
        if move:
            shutil.move(src_path, dest_path)
        else:
            shutil.copy2(src_path, dest_path)

    print(f"{'Moved' if move else 'Copied'} {len(selected_files)} images from {src_dir} → {dest_dir}")

In [20]:
# Copy 100 random images from class1/date1 to validation folder
split_images(
    src_dir = '/Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Train/Peganum harmala L./04-01-2025 (85)',
    dest_dir = "/Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Validation/Peganum harmala L./04-01-2025 (22)",
    num_images=22,
    move=True   # set True if you want to move instead of copy
)

Moved 22 images from /Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Train/Peganum harmala L./04-01-2025 (85) → /Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Validation/Peganum harmala L./04-01-2025 (22)


In [38]:
# Copy 100 random images from class1/date1 to validation folder
split_images(
    src_dir = '/Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Train/Iphiona mucronata (Forssk.) Asch. & Schweinf./31-01-2025 (32)',
    dest_dir = "/Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Validation/Iphiona mucronata (Forssk.) Asch. & Schweinf./31-01-2025 (28)",
    num_images=28,
    move=True   # set True if you want to move instead of copy
)

Moved 28 images from /Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Train/Iphiona mucronata (Forssk.) Asch. & Schweinf./31-01-2025 (32) → /Users/drmorsy/Downloads/Wadi Degla/Plants/Plant_Subset/Validation/Iphiona mucronata (Forssk.) Asch. & Schweinf./31-01-2025 (28)


## Dataset metadata extraction

In [17]:
def parse_folder_structure(base_path):
    data = []
    
    for species_folder in os.listdir(base_path):
        species_path = os.path.join(base_path, species_folder)
        if not os.path.isdir(species_path):
            continue


        binomial_name = species_folder

        for date_folder in os.listdir(species_path):
            date_path = os.path.join(species_path, date_folder)
            if not os.path.isdir(date_path):
                continue
                
            # Parse date and location from date folder name
            date_match = re.match(r'(\d{1,2}-\d{1,2}-\d{4})', date_folder)
            location_match = re.search(r'K\d+', date_folder)

            date = date_match.group(1) if date_match else None
            location = location_match.group(0) if location_match else None

            # Count the number of visible image files in the date folder
            image_count = sum(
                1 for item in os.listdir(date_path)
                if os.path.isfile(os.path.join(date_path, item)) and not item.startswith('.')
            )

            # Append data to the list
            data.append({
                "Binomial Name": binomial_name,
                "Date": date,
                "Number of Images": image_count,
                "Location": location
            })

    # Create a DataFrame from the data
    df = pd.DataFrame(data)
    return df

# Set the base path to the root directory (where species folders are now directly stored)
train_path = "/Users/drmorsy/Downloads/Wadi Degla/Plants/Pilot_Subset/Train"
val_path = "/Users/drmorsy/Downloads/Wadi Degla/Plants/Pilot_Subset/Validation"

# Generate the DataFrame
TrainSpeciesDataset = parse_folder_structure(train_path)
ValSpeciesDataset = parse_folder_structure(val_path)

In [33]:
ValSpeciesDataset

,Binomial Name,Date,Number of Images,Location
0,Ochradenus baccatus Delile,08-03-2024,498,None
1,Diplotaxis harra (Forssk.) Boiss.,31-01-2025,136,None
2,Capparis spinosa L.,04-01-2025,142,K12
3,Peganum harmala L.,04-01-2025,22,None
4,Tamarix nilotica (Ehrenb.) Bunge,04-01-2025,187,K12
5,Tamarix nilotica (Ehrenb.) Bunge,20-01-2024,133,K12
6,Iphiona mucronata (Forssk.) Asch. & Schweinf.,31-01-2025,28,None


## Data Integrity and Split Validation

In [46]:
train_dates = TrainSpeciesDataset.groupby('Binomial Name')['Date'].agg(lambda x: ', '.join(sorted(x.unique())))
val_dates   = ValSpeciesDataset.groupby('Binomial Name')['Date'].agg(lambda x: ', '.join(sorted(x.unique())))

summary_table = pd.merge(
    TrainSpeciesDataset.groupby('Binomial Name')[['Number of Images']].sum(),
    ValSpeciesDataset.groupby('Binomial Name')[['Number of Images']].sum(),
    left_index=True,
    right_index=True,
    suffixes=('_Train', '_Val')
)

# Add date columns
summary_table = summary_table.merge(train_dates, left_index=True, right_index=True, how='left')
summary_table = summary_table.merge(val_dates, left_index=True, right_index=True, how='left', suffixes=('_Train_Date', '_Val_Date'))
# Rename date columns clearly
summary_table.rename(columns={'Date_Train_Date': 'Train_Dates'}, inplace=True)
summary_table.rename(columns={'Date_Val_Date': 'Val_Dates'}, inplace=True)

# Add total + percentages
summary_table['Total'] = summary_table['Number of Images_Train'] + summary_table['Number of Images_Val']
summary_table['Train_percentage'] = (summary_table['Number of Images_Train'] / summary_table['Total']) * 100
summary_table['Val_percentage']   = (summary_table['Number of Images_Val']   / summary_table['Total']) * 100


totals = pd.DataFrame({
    'Number of Images_Train': [summary_table['Number of Images_Train'].sum()],
    'Number of Images_Val': [summary_table['Number of Images_Val'].sum()],
    'Train_Dates': ['–'],
    'Val_Dates': ['–'],
    'Total': [summary_table['Total'].sum()]
}, index=['All'])

totals['Train_percentage'] = (totals['Number of Images_Train'] / totals['Total']) * 100
totals['Val_percentage']   = (totals['Number of Images_Val']   / totals['Total']) * 100

# Append
summary_table = pd.concat([summary_table, totals])
pd.set_option('display.max_colwidth', None)
summary_table

,Number of Images_Train,Number of Images_Val,Train_Dates,Val_Dates,Total,Train_percentage,Val_percentage
Capparis spinosa L.,515,142,"21-02-2025, 22-11-2024, 22-12-2023, 23-12-2024, 31-01-2025",04-01-2025,657,78.386606,21.613394
Diplotaxis harra (Forssk.) Boiss.,415,136,"04-01-2025, 08-03-2024, 22-11-2024",31-01-2025,551,75.317604,24.682396
Iphiona mucronata (Forssk.) Asch. & Schweinf.,111,28,"23-12-2024, 31-01-2025",31-01-2025,139,79.856115,20.143885
Ochradenus baccatus Delile,1396,498,"04-01-2025, 20-01-2024, 22-11-2024, 22-12-2023, 23-12-2024, 31-01-2025",08-03-2024,1894,73.706441,26.293559
Peganum harmala L.,85,22,04-01-2025,04-01-2025,107,79.439252,20.560748
Tamarix nilotica (Ehrenb.) Bunge,1074,320,"08-03-2024, 21-02-2025, 22-11-2024, 22-12-2023, 23-12-2024, 31-01-2025","04-01-2025, 20-01-2024",1394,77.044476,22.955524
All,3596,1146,–,–,4742,75.832982,24.167018


| Class                                   | Train_dates                                                                 | Validation_dates     | Train_num_images | Val_num_images | Train_percentage | Val_percentage |
|-----------------------------------------|----------------------------------------------------------------------------|----------------------|------------------|----------------|------------------|----------------|
| Capparis spinosa L.                     | 2025-01-31 ,2024-11-22 ,2023-12-22 ,2024-12-23 ,2025-02-21                 | 2025-01-04           | 515              | 142            | 0.783866058      | 0.216133942    |
| Diplotaxis harra (Forssk.) Boiss.       | 2024-11-22 ,2024-03-08 ,2025-01-04                                         | 2025-01-31           | 415              | 136            | 0.753176044      | 0.246823956    |
| Iphiona mucronata (Forssk.) Asch. & Schweinf. | 2025-01-31 (32) ,2024-12-23                                                 | 2025-01-31 (28)      | 111              | 28             | 0.798561151      | 0.201438849    |
| Ochradenus baccatus Delile              | 2024-11-22 ,2024-12-23 ,2025-01-04 ,2023-12-22 ,2025-01-31 ,2024-01-20     | 2024-03-08           | 1396             | 498            | 0.737064414      | 0.262935586    |
| Peganum harmala L.                      | 2025-01-04 (85)                                                            | 2025-01-04 (22)      | 85               | 22             | 0.794392523      | 0.205607477    |
| Tamarix nilotica (Ehrenb.) Bunge        | 2024-11-22 ,2024-12-23 ,2023-12-22 ,2025-02-21 ,2025-01-31 ,2024-03-08     | 2024-01-20 ,2025-01-04 | 1074             | 320            | 0.770444763      | 0.229555237    |
| **Total**                               |                                                                            |                      | **3596**         | **1146**       | **0.758329819**  | **0.241670181** |

## Exploratory Data Analysis (EDA) of dataset splits

# <center>**Input Pipeline Setup (Dataset Loading & Transformations)**</center>

## **1. Experiment Configuration**

In [334]:
DATA_DIR = Path("/Users/drmorsy/Downloads/Wadi Degla/Plants/Pilot_Subset")
BATCH_SIZE = 32
EPOCHS = 10
LR = 0.001
NUM_CLASSES = sum(1 for d in (DATA_DIR/'Train').iterdir() if d.is_dir())

## **2. Define Data Augmentation & Preprocessing Pipeline**

**Resize then Augment**
- **When working with plant images of varying sizes, the recommended order is: resize first, then augment.**


1. **Consistency:** Resizing first ensures all images share the same dimensions, leading to consistent transformation behavior during augmentation.

2. **Computational Efficiency:** Augmentation is faster on smaller, uniform images, while large images use more memory and slow training.
  
3. **Avoids Distortion Cascading:** Resizing first provides a clean base, preventing amplified artifacts (blur, aliasing, edge blanks) that occur when resizing after augmentation.
  

In [335]:
# Optimized Plant Identification Augmentation Pipeline for Training
# Define the transformation pipeline for training (with augmentation)

train_transforms = v2.Compose([
    # -----------------------------------------------------------
    # Stage 1: Input Preparation & Spatial Augmentations
    # -----------------------------------------------------------
    v2.ToImage(),  # Convert PIL/HWC to torch.Tensor(CHW, uint8)
    
    # Core spatial augmentations - applied in uint8 for speed/quality
    v2.RandomResizedCrop(
        size=(224, 224), 
        scale=(0.2, 1.0),
        ratio=(0.75, 1.33),
        antialias=True
    ),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.2),
    v2.RandomApply([
    v2.RandomRotation(
        degrees=(-90, 90), 
        interpolation=v2.InterpolationMode.BILINEAR
    )
], p=0.5),
    
    # -----------------------------------------------------------
    # Stage 2: Color Operations
    # -----------------------------------------------------------
    v2.ToDtype(torch.float32, scale=True),  # Convert to [0,1] float32
    
    # Apply color jitter randomly (80% of the time)
    v2.RandomApply([
        v2.ColorJitter(
            brightness=(0.7, 1.3),
            contrast=(0.85, 1.3),
            saturation=(0.7, 1.3),
            hue=(-0.0278, 0.0278)
        )
    ], p=0.8),  # Apply color jitter 80% of the time, identity 20%
    
    # -----------------------------------------------------------
    # Stage 3: Normalization (CRUCIAL for training)
    # -----------------------------------------------------------
    v2.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

**In validation, it is best to first resize the image so that its shorter side is 256 pixels—whether by upscaling or downscaling—and then apply a center crop of 224×224 to obtain a well-centered, standardized region.**

In [336]:
# Validation/Test transforms (no augmentation)
val_transforms = v2.Compose([
    v2.ToImage(),
    # Resize the image so its shorter side is 256 pixels, preserving aspect ratio (i.e., the proportion between width and height stays the same)
    v2.Resize(256),
    # Extract a 224x224 patch from the center of the resized image
    v2.CenterCrop(224),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## **3. Dataset Construction**

### **3.1 File Extension Validation**

In [337]:
def count_extensions(root_dir):
    extension_counter = defaultdict(int)
    c = 0
    for foldername, subfolders, filenames in os.walk(root_dir):
      
            
        for filename in filenames:
            # Extract extension and convert to lowercase
            ext = os.path.splitext(filename)[1].lower()
            if ext == '':
                continue
            # Count extension (even if empty)
            extension_counter[ext] += 1
    
    return dict(extension_counter)

# Usage
extension_counts = count_extensions(DATA_DIR)

print("Extension counts:")
for ext, count in extension_counts.items():
    print(f"{ext}: {count}")
print(f'Total: {sum(extension_counts.values())}')    

Extension counts:
.jpg: 4081
.jpeg: 661
Total: 4742


### **3.2 Dataset creation**

In [338]:
# Create Dataset objects for training and validation sets

train_dataset = datasets.ImageFolder(
    root=DATA_DIR / "Train",  # Path to the training directory
    transform=train_transforms  # Apply the aggressive augmentation pipeline
)

val_dataset = datasets.ImageFolder(
    root=DATA_DIR / "Validation",  # Path to the validation directory
    transform=val_transforms  # Apply the deterministic preprocessing pipeline
)

print(f"Training dataset size: {len(train_dataset)} images")
print(f"Validation dataset size: {len(val_dataset)} images")
print(f"Dataset classes: {train_dataset.classes}")

Training dataset size: 3596 images
Validation dataset size: 1146 images
Dataset classes: ['Capparis spinosa L.', 'Diplotaxis harra (Forssk.) Boiss.', 'Iphiona mucronata (Forssk.) Asch. & Schweinf.', 'Ochradenus baccatus Delile', 'Peganum harmala L.', 'Tamarix nilotica (Ehrenb.) Bunge']


## **DataLoader Configuration**

**This is where we wrap the datasets into DataLoaders for batching, shuffling, and parallel loading.**

In [339]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,        # Crucial for training: shuffles data each epoch
    num_workers=4,       # Number of subprocesses for data loading
    pin_memory=True      # Speeds up data transfer to GPU (if available)
)

In [340]:
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,       # No need to shuffle validation data
    num_workers=4,
    pin_memory=True
)

In [341]:
# Verify the loaders
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 113
Validation batches: 36


In [342]:
len(next(iter(train_loader))[0])

32

# <center>**Utility Functions for Model Training & Evaluation**</center>

## **1. Compute Class Weights**

In [67]:
def compute_class_weights(dataset, method="inverse_frequency", normalize=False):
    """
    Compute class weights based on the chosen method, with optional normalization.

    Args:
        dataset: A dataset object with a 'targets' attribute containing class labels.
        method (str): Method for computing class weights. Options:
            - "inverse_frequency": total_samples / count
            - "balanced": total_samples / (count * num_classes)
            - "sqrt_inverse": 1 / np.sqrt(count)
            - "scaled_sqrt_inverse": (1 / np.sqrt(count)) * 100
        normalize (bool): If True, rescale weights so that their mean = 1.

    Returns:
        torch.Tensor: Class weights tensor on the appropriate device.
    """
    # Compute class frequencies
    class_counts = Counter(dataset.targets)
    total_samples = sum(class_counts.values())
    num_classes = len(class_counts)

    # Select weight calculation method
    if method == "inverse_frequency":
        class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
    elif method == "balanced":
        class_weights = {cls: total_samples / (count * num_classes) for cls, count in class_counts.items()}
    elif method == "sqrt_inverse":
        class_weights = {cls: 1 / np.sqrt(count) for cls, count in class_counts.items()}
    elif method == "scaled_sqrt_inverse":
        class_weights = {cls: (1 / np.sqrt(count)) * 100 for cls, count in class_counts.items()}
    else:
        raise ValueError("Invalid method. Choose from 'inverse_frequency', 'balanced', 'sqrt_inverse', 'scaled_sqrt_inverse'.")

    # Convert to tensor
    class_weights_tensor = torch.tensor([class_weights[i] for i in range(num_classes)], dtype=torch.float32)

    #  Normalize weights to have an average of 1.0 → divide by mean 
    if normalize:
        class_weights_tensor = class_weights_tensor / class_weights_tensor.mean()

    # Move weights to MPS (Apple GPU) if available
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    return class_weights_tensor.to(device)

In [68]:
class_index_count = {train_dataset.classes[i]:sorted(Counter(train_dataset.targets).items())[i] for i in range(len(train_dataset.classes))}
print(class_index_count)

{'Capparis spinosa L.': (0, 515), 'Diplotaxis harra (Forssk.) Boiss.': (1, 415), 'Iphiona mucronata (Forssk.) Asch. & Schweinf.': (2, 111), 'Ochradenus baccatus Delile': (3, 1396), 'Peganum harmala L.': (4, 85), 'Tamarix nilotica (Ehrenb.) Bunge': (5, 1074)}


In [69]:
inverse_frequency_class_weights = compute_class_weights(train_dataset, method="inverse_frequency")
normalized_inverse_frequency_class_weights = compute_class_weights(train_dataset, method="inverse_frequency", normalize=True)

In [84]:
print(f"Inverse frequency class weights: {[round(v, 2) for v in inverse_frequency_class_weights.cpu().numpy().tolist()]}")
print(f"Inverse frequency class weights: {round(inverse_frequency_class_weights.mean().item(), 2)}")


Inverse frequency class weights: [6.98, 8.67, 32.4, 2.58, 42.31, 3.35]
Inverse frequency class weights: 16.05


In [86]:
print(f"Normalized Inverse frequency class weights: {[round(v, 2) for v in normalized_inverse_frequency_class_weights.cpu().numpy().tolist()]}")
print(f"Normalized Inverse frequency class weights: {round(normalized_inverse_frequency_class_weights.mean().item(), 2)}")

Normalized Inverse frequency class weights: [0.44, 0.54, 2.02, 0.16, 2.64, 0.21]
Normalized Inverse frequency class weights: 1.0


## **2. Load Candidate Models**

In [408]:
def load_mobilenet_v2(dataset, num_unfreeze=20, device=None):
    """
    Load a pretrained MobileNetV2 model, freeze layers except the last `num_unfreeze` layers,
    and replace the classifier for a custom number of classes.

    Args:
        dataset: Dataset object with a 'classes' attribute to determine the number of output classes.
        num_unfreeze (int): Number of layers to keep trainable from the end.
        device (torch.device, optional): Device to move the model to (defaults to MPS or CPU).

    Returns:
        nn.Module: Modified MobileNetV2 model on specified device
    """
    # Set device if not provided
    if device is None:
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    # Load pretrained MobileNetV2
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    

    # Freeze all layers except the last `num_unfreeze` trainable layers
    for param in list(model.features.parameters())[:-num_unfreeze]:
        param.requires_grad = False

    # Replace classifier to match dataset classes
    num_classes = len(dataset.classes)
    model.classifier = nn.Sequential(
        nn.Dropout(0.2),  # Regularization
        nn.Linear(model.last_channel, num_classes)
    )

    # Model summary
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}\nTrainable parameters: {trainable_params:,}\nPercentage of trainable parameters: {(trainable_params/total_params)*100:.2f}%")
    print(f"Model moved to: {device}")

    return model.to(device)

## 3. Model Training

In [409]:
def train_model(model, train_loader, criterion, optimizer, class_names, epoch_num=1, log_interval=100):
    start_time = time.time()  # Start tracking time
    device = torch.device("cuda" if torch.cuda.is_available() else "mps")  # Use GPU if available on Mac

    # Dictionary to store per-class metrics
    class_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "y_true": [], "y_pred": []})

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start_time = time.time()  # Track epoch time

    for batch_idx, (images, labels) in enumerate(train_loader, 1):  # Start index from 1
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Compute accuracy
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        accuracy = 100 * correct / total

        # Store predictions for per-class metrics
        for label, pred in zip(labels.cpu().numpy(), predicted.cpu().numpy()):
            class_metrics[label]["correct"] += (label == pred)
            class_metrics[label]["total"] += 1
            class_metrics[label]["y_true"].append(label)
            class_metrics[label]["y_pred"].append(pred)

        # Print loss and accuracy every 100 batches
        if batch_idx % log_interval == 0:
            elapsed_time = (time.time() - start_time)/60
            print(f"Epoch {epoch_num}, Batch {batch_idx}, Loss: {running_loss/batch_idx:.4f}, "
                  f"Accuracy: {accuracy:.2f}%, Time Passed: {elapsed_time:.2f}m")

    epoch_time = (time.time() - epoch_start_time)/60
    print(f"Epoch {epoch_num} Completed - Average Loss: {running_loss/len(train_loader):.4f}, "
          f"Accuracy: {accuracy:.2f}%, Epoch Time: {epoch_time:.2f}m")
    overall_accuracy = 100 * correct / total

    total_time = (time.time() - start_time)/60
    print(f"Training Complete Epoch {epoch_num} - Total Time: {total_time:.2f}m")

    # Calculate per-class accuracy, precision, recall, and F1-score
    all_y_true = []
    all_y_pred = []
    
    for class_idx in sorted(class_metrics.keys()):
        all_y_true.extend(class_metrics[class_idx]["y_true"])
        all_y_pred.extend(class_metrics[class_idx]["y_pred"])
    
    # Compute per-class precision, recall, and F1-score
    unique_classes = sorted(set(all_y_true))  # Get all unique classes
    per_class_metrics = {
        "precision": precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "recall": recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "f1_score": f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
    }

    # Store per-class results in alignment with unique_classes
    class_results = []
    for class_idx in unique_classes:
        class_results.append([
            class_names[class_idx],
            per_class_metrics["precision"][unique_classes.index(class_idx)]*100,
            per_class_metrics["recall"][unique_classes.index(class_idx)]*100,
            per_class_metrics["f1_score"][unique_classes.index(class_idx)]*100
        ])

    # Create a DataFrame with per-class accuracy, precision, recall, and F1-score
    df_results = pd.DataFrame(class_results, columns=["Species", "Precision", "Recall", "F1-score"])

    # Append overall metrics (Macro) as the last row
    macro_precision = per_class_metrics["precision"].mean()*100
    macro_recall = per_class_metrics["recall"].mean()*100
    macro_f1 = per_class_metrics["f1_score"].mean()*100

    df_results.loc[len(df_results)] = ["Average Performance (Macro)", macro_precision, macro_recall, macro_f1]

    df_results['Epoch'] = epoch_num
    avg_loss = running_loss / len(train_loader)

    return {
    'model': model,
    'optimizer': optimizer,
    'metrics_df': df_results,
    'overall_accuracy': overall_accuracy,
    'epoch_loss': running_loss,
    'avg_loss': avg_loss,
    'epoch_time': epoch_time
            }

## 4. Model Evaluation

In [410]:
def evaluate_model(model, val_loader, criterion, class_names, epoch_num=1, log_interval=100):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "mps")
    
    start_time = time.time()  # Track evaluation time

    # Dictionary to store per-class metrics
    class_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "y_true": [], "y_pred": []})
    all_y_true = []
    all_y_pred = []
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(val_loader, 1):  # Start index from 1
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            # Compute loss
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)

            all_y_true.extend(labels.cpu().numpy())
            all_y_pred.extend(preds.cpu().numpy())

            # Compute accuracy
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            accuracy = 100 * correct / total

            # Store per-class metrics
            for label, pred in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                class_metrics[label]["correct"] += (label == pred)
                class_metrics[label]["total"] += 1
                class_metrics[label]["y_true"].append(label)
                class_metrics[label]["y_pred"].append(pred)

            # Print progress every 100 batches
            if batch_idx % log_interval == 0:
                elapsed_time = (time.time() - start_time)/60
                avg_loss = running_loss / batch_idx
                print(f"Batch {batch_idx}, Loss: {avg_loss:.4f}, "
                      f"Accuracy: {accuracy:.2f}%, Time Passed: {elapsed_time:.2f}m")

    total_time = (time.time() - start_time)/60
    avg_loss = running_loss / len(val_loader)
    print(f"Evaluation Complete - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%, Total Time: {total_time:.2f}m")
    overall_accuracy = 100 * correct / total

    # Compute per-class accuracy
    class_results = []
    for class_idx in sorted(class_metrics.keys()):
        metrics = class_metrics[class_idx]
        class_results.append([class_names[class_idx]])

    # Compute per-class precision, recall, and F1-score
    unique_classes = sorted(set(all_y_true))  # Get all unique classes
    per_class_metrics = {
        "precision": precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "recall": recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0),
        "f1_score": f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
    }

    # Store per-class metrics
    for idx, class_label in enumerate(unique_classes):
        class_results[idx].extend([
            per_class_metrics["precision"][idx],
            per_class_metrics["recall"][idx],
            per_class_metrics["f1_score"][idx]
        ])

    # Create a DataFrame with per-class accuracy, precision, recall, and F1-score
    df_results = pd.DataFrame(class_results, columns=["Species", "Precision", "Recall", "F1-score"])

    # Append overall metrics (Macro) as the last row
    macro_precision = per_class_metrics["precision"].mean()
    macro_recall = per_class_metrics["recall"].mean()
    macro_f1 = per_class_metrics["f1_score"].mean()

    df_results.loc[len(df_results)] = ["Average Performance (Macro)", macro_precision, macro_recall, macro_f1]

    df_results['Epoch'] = epoch_num
    return {
    'metrics_df': df_results,
    'overall_accuracy': overall_accuracy,
    'avg_loss': avg_loss,
    'epoch_time': total_time
            }

## 5. Create performance summary

In [432]:
def create_performance_summary(df, micro_accuracy, avg_loss, data_loader, model_name, train_val, class_weights, time_taken):
    # Extract macro metrics from the last row
    
    macro_row = df[df['Species'] == 'Average Performance (Macro)']
    precision_macro = macro_row['Precision'].values[0]
    recall_macro = macro_row['Recall'].values[0]
    f1_macro = macro_row['F1-score'].values[0]


    # Exclude the last row to analyze per-class accuracies
    class_accuracies = df[df['Species'] != 'Average Performance (Macro)'].copy()

    # Identify best and worst class based on accuracy
    best_class = class_accuracies.loc[class_accuracies['F1-score'].idxmax(), 'Species']
    worst_class = class_accuracies.loc[class_accuracies['F1-score'].idxmin(), 'Species']

    # Compute the median accuracy and find the nearest class
    median_f1_macro = class_accuracies['F1-score'].median()
    class_accuracies['Abs_Diff'] = (class_accuracies['F1-score'] - median_f1_macro).abs()
    median_class = class_accuracies.loc[class_accuracies['Abs_Diff'].idxmin(), 'Species']

    # Compute the number of classes above micro and macro accuracy averages
    num_classes_above_f1_macro_avg = (class_accuracies['F1-score'] > f1_macro).sum()
    num_classes_above_f1_macro_median = (class_accuracies['F1-score'] > median_f1_macro).sum()

    # Create the final summary DataFrame
    performance_summary = pd.DataFrame({
        "Model": [model_name],
        "Train & Test": [train_val],
        "Accuracy (micro)": [micro_accuracy],
        "Precision (macro)": [precision_macro],
        "Recall (macro)": [recall_macro],
        "F1 Score": [f1_macro],
        "Loss": [avg_loss],
        "Class_weights": [class_weights],
        "Best_class": [best_class],
        "Median_class": [median_class],
        "Worst_class": [worst_class],
        "num_classes_above_f1_macro_avg": [num_classes_above_f1_macro_avg],
        "num_classes_above_f1_macro_median": [num_classes_above_f1_macro_median],
        "Time Taken (mins)": [time_taken]
    })


    return performance_summary

# Training

In [443]:
model = load_mobilenet_v2(train_dataset, num_unfreeze=20)

Total parameters: 2,231,558
Trainable parameters: 1,060,166
Percentage of trainable parameters: 47.51%
Model moved to: mps


In [444]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adamax(model.parameters(), lr=0.02)

In [445]:
train_dataset.transforms

StandardTransform
Transform: Compose(
                 ToImage()
                 RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33), interpolation=InterpolationMode.BILINEAR, antialias=True)
                 RandomHorizontalFlip(p=0.5)
                 RandomVerticalFlip(p=0.2)
                 RandomApply(    RandomRotation(degrees=[-90.0, 90.0], interpolation=InterpolationMode.BILINEAR, expand=False, fill=0))
                 ToDtype(scale=True)
                 RandomApply(    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3), saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)))
                 Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
           )

In [446]:
val_dataset.transforms

StandardTransform
Transform: Compose(
                 ToImage()
                 Resize(size=[256], interpolation=InterpolationMode.BILINEAR, antialias=True)
                 CenterCrop(size=(224, 224))
                 ToDtype(scale=True)
                 Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
           )

In [447]:
# Initialize lists to store data across epochs
train_results_list = []
train_summary_list = []
val_results_list = []
val_summary_list = []

for epoch in range(1, 11):
    print(f"\n🚀 Epoch {epoch} - Training Started...\n")
    
    # Training Phase
    training_result = train_model(model, train_loader, criterion, optimizer, train_dataset.classes, epoch_num=epoch, log_interval=20)
    
    train_summary = create_performance_summary(
        training_result['metrics_df'], training_result['overall_accuracy'], training_result['avg_loss'], train_loader,
        model_name='mobilenet_v2', train_val='Train', class_weights='None',
        time_taken=training_result['epoch_time']
    )

    
    # Append training results
    train_results_list.append(training_result['metrics_df'])
    train_summary_list.append(train_summary)

    print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
    # Evaluation Phase
    val_result = evaluate_model(model, val_loader, criterion, val_dataset.classes, epoch_num=epoch, log_interval=20)

    val_summary = create_performance_summary(
        val_result['metrics_df'], val_result['overall_accuracy'], val_result['avg_loss'], val_loader,
        model_name='mobilenet_v2', train_val='Test', class_weights='None',
        time_taken=val_result['epoch_time']
    )
    
    # Append valing results
    val_results_list.append(val_result['metrics_df'])
    val_summary_list.append(val_summary)

    print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

    # Convert lists to DataFrames
    train_results_full = pd.concat(train_results_list, ignore_index=True)
    train_results_full.to_csv("train_results_full.csv", index=False)
    
    train_summary_full = pd.concat(train_summary_list, ignore_index=True)
    train_summary_full.to_csv("train_summary_full.csv", index=False)
    
    val_results_full = pd.concat(val_results_list, ignore_index=True)
    val_results_full.to_csv("val_results_full.csv", index=False)
    
    val_summary_full = pd.concat(val_summary_list, ignore_index=True)
    val_summary_full.to_csv("val_summary_full.csv", index=False)


🚀 Epoch 1 - Training Started...

Epoch 1, Batch 20, Loss: 1.0073, Accuracy: 68.12%, Time Passed: 0.74m
Epoch 1, Batch 40, Loss: 0.8189, Accuracy: 72.89%, Time Passed: 1.42m
Epoch 1, Batch 60, Loss: 0.6854, Accuracy: 77.29%, Time Passed: 2.21m
Epoch 1, Batch 80, Loss: 0.6222, Accuracy: 79.30%, Time Passed: 3.00m
Epoch 1, Batch 100, Loss: 0.5789, Accuracy: 80.78%, Time Passed: 3.71m
Epoch 1 Completed - Average Loss: 0.5569, Accuracy: 81.62%, Epoch Time: 4.12m
Training Complete Epoch 1 - Total Time: 4.12m

✅ Epoch 1 - Training Completed. Starting Evaluation...

Batch 20, Loss: 0.7612, Accuracy: 85.00%, Time Passed: 1.06m
Evaluation Complete - Loss: 0.7173, Accuracy: 80.02%, Total Time: 2.07m

📊 Epoch 1 - Evaluation Completed.


🚀 Epoch 2 - Training Started...

Epoch 2, Batch 20, Loss: 0.3542, Accuracy: 88.28%, Time Passed: 0.69m
Epoch 2, Batch 40, Loss: 0.3589, Accuracy: 88.36%, Time Passed: 1.42m
Epoch 2, Batch 60, Loss: 0.3378, Accuracy: 88.75%, Time Passed: 2.16m
Epoch 2, Batch 80, Lo

In [507]:
model_10 = model

In [510]:
# Initialize lists to store data across epochs
train_results_list = []
train_summary_list = []
val_results_list = []
val_summary_list = []

for epoch in range(11, 21):
    print(f"\n🚀 Epoch {epoch} - Training Started...\n")
    
    # Training Phase
    training_result = train_model(model, train_loader, criterion, optimizer, train_dataset.classes, epoch_num=epoch, log_interval=20)
    
    train_summary = create_performance_summary(
        training_result['metrics_df'], training_result['overall_accuracy'], training_result['avg_loss'], train_loader,
        model_name='mobilenet_v2', train_val='Train', class_weights='None',
        time_taken=training_result['epoch_time']
    )

    
    # Append training results
    train_results_list.append(training_result['metrics_df'])
    train_summary_list.append(train_summary)

    print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
    # Evaluation Phase
    val_result = evaluate_model(model, val_loader, criterion, val_dataset.classes, epoch_num=epoch, log_interval=20)

    val_summary = create_performance_summary(
        val_result['metrics_df'], val_result['overall_accuracy'], val_result['avg_loss'], val_loader,
        model_name='mobilenet_v2', train_val='Test', class_weights='None',
        time_taken=val_result['epoch_time']
    )
    
    # Append valing results
    val_results_list.append(val_result['metrics_df'])
    val_summary_list.append(val_summary)

    print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

    # Convert lists to DataFrames
    train_results_full_20 = pd.concat(train_results_list, ignore_index=True)
    train_results_full_20.to_csv("train_results_full_20.csv", index=False)
    
    train_summary_full_20 = pd.concat(train_summary_list, ignore_index=True)
    train_summary_full_20.to_csv("train_summary_full_20.csv", index=False)
    
    val_results_full_20 = pd.concat(val_results_list, ignore_index=True)
    val_results_full_20.to_csv("val_results_full_20.csv", index=False)
    
    val_summary_full_20 = pd.concat(val_summary_list, ignore_index=True)
    val_summary_full_20.to_csv("val_summary_full_20.csv", index=False)


🚀 Epoch 11 - Training Started...

Epoch 11, Batch 20, Loss: 0.1015, Accuracy: 96.88%, Time Passed: 0.71m
Epoch 11, Batch 40, Loss: 0.0904, Accuracy: 97.19%, Time Passed: 1.29m
Epoch 11, Batch 60, Loss: 0.1003, Accuracy: 96.98%, Time Passed: 1.89m
Epoch 11, Batch 80, Loss: 0.0994, Accuracy: 96.84%, Time Passed: 2.49m
Epoch 11, Batch 100, Loss: 0.1040, Accuracy: 96.78%, Time Passed: 3.23m
Epoch 11 Completed - Average Loss: 0.1051, Accuracy: 96.83%, Epoch Time: 3.65m
Training Complete Epoch 11 - Total Time: 3.65m

✅ Epoch 11 - Training Completed. Starting Evaluation...

Batch 20, Loss: 0.4299, Accuracy: 87.66%, Time Passed: 2.56m
Evaluation Complete - Loss: 0.3011, Accuracy: 90.66%, Total Time: 3.54m

📊 Epoch 11 - Evaluation Completed.


🚀 Epoch 12 - Training Started...

Epoch 12, Batch 20, Loss: 0.0704, Accuracy: 97.97%, Time Passed: 0.70m
Epoch 12, Batch 40, Loss: 0.0790, Accuracy: 97.89%, Time Passed: 1.34m
Epoch 12, Batch 60, Loss: 0.1011, Accuracy: 96.82%, Time Passed: 2.10m
Epoch 1

In [511]:
model_20 = model

In [512]:
# Initialize lists to store data across epochs
train_results_list = []
train_summary_list = []
val_results_list = []
val_summary_list = []

for epoch in range(11, 21):
    print(f"\n🚀 Epoch {epoch} - Training Started...\n")
    
    # Training Phase
    training_result = train_model(model, train_loader, criterion, optimizer, train_dataset.classes, epoch_num=epoch, log_interval=20)
    
    train_summary = create_performance_summary(
        training_result['metrics_df'], training_result['overall_accuracy'], training_result['avg_loss'], train_loader,
        model_name='mobilenet_v2', train_val='Train', class_weights='None',
        time_taken=training_result['epoch_time']
    )

    
    # Append training results
    train_results_list.append(training_result['metrics_df'])
    train_summary_list.append(train_summary)

    print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
    # Evaluation Phase
    val_result = evaluate_model(model, val_loader, criterion, val_dataset.classes, epoch_num=epoch, log_interval=20)

    val_summary = create_performance_summary(
        val_result['metrics_df'], val_result['overall_accuracy'], val_result['avg_loss'], val_loader,
        model_name='mobilenet_v2', train_val='Test', class_weights='None',
        time_taken=val_result['epoch_time']
    )
    
    # Append valing results
    val_results_list.append(val_result['metrics_df'])
    val_summary_list.append(val_summary)

    print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

    # Convert lists to DataFrames
    train_results_full_30 = pd.concat(train_results_list, ignore_index=True)
    train_results_full_30.to_csv("train_results_full_30.csv", index=False)
    
    train_summary_full_30 = pd.concat(train_summary_list, ignore_index=True)
    train_summary_full_30.to_csv("train_summary_full_30.csv", index=False)
    
    val_results_full_30 = pd.concat(val_results_list, ignore_index=True)
    val_results_full_30.to_csv("val_results_full_30.csv", index=False)
    
    val_summary_full_30 = pd.concat(val_summary_list, ignore_index=True)
    val_summary_full_30.to_csv("val_summary_full_30.csv", index=False)


🚀 Epoch 11 - Training Started...

Epoch 11, Batch 20, Loss: 0.0692, Accuracy: 97.50%, Time Passed: 0.67m
Epoch 11, Batch 40, Loss: 0.0706, Accuracy: 97.66%, Time Passed: 1.35m
Epoch 11, Batch 60, Loss: 0.0681, Accuracy: 97.71%, Time Passed: 2.01m
Epoch 11, Batch 80, Loss: 0.0654, Accuracy: 97.81%, Time Passed: 2.72m
Epoch 11, Batch 100, Loss: 0.0623, Accuracy: 97.91%, Time Passed: 3.45m
Epoch 11 Completed - Average Loss: 0.0656, Accuracy: 97.89%, Epoch Time: 3.92m
Training Complete Epoch 11 - Total Time: 3.92m

✅ Epoch 11 - Training Completed. Starting Evaluation...

Batch 20, Loss: 0.3274, Accuracy: 93.12%, Time Passed: 1.11m
Evaluation Complete - Loss: 0.3154, Accuracy: 93.46%, Total Time: 2.13m

📊 Epoch 11 - Evaluation Completed.


🚀 Epoch 12 - Training Started...

Epoch 12, Batch 20, Loss: 0.0828, Accuracy: 97.19%, Time Passed: 0.78m
Epoch 12, Batch 40, Loss: 0.0630, Accuracy: 97.97%, Time Passed: 1.48m
Epoch 12, Batch 60, Loss: 0.0581, Accuracy: 98.23%, Time Passed: 2.22m
Epoch 1

In [513]:
model_30 = model

In [514]:
# Initialize lists to store data across epochs
train_results_list = []
train_summary_list = []
val_results_list = []
val_summary_list = []

for epoch in range(31, 41):
    print(f"\n🚀 Epoch {epoch} - Training Started...\n")
    
    # Training Phase
    training_result = train_model(model, train_loader, criterion, optimizer, train_dataset.classes, epoch_num=epoch, log_interval=20)
    
    train_summary = create_performance_summary(
        training_result['metrics_df'], training_result['overall_accuracy'], training_result['avg_loss'], train_loader,
        model_name='mobilenet_v2', train_val='Train', class_weights='None',
        time_taken=training_result['epoch_time']
    )

    
    # Append training results
    train_results_list.append(training_result['metrics_df'])
    train_summary_list.append(train_summary)

    print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
    # Evaluation Phase
    val_result = evaluate_model(model, val_loader, criterion, val_dataset.classes, epoch_num=epoch, log_interval=20)

    val_summary = create_performance_summary(
        val_result['metrics_df'], val_result['overall_accuracy'], val_result['avg_loss'], val_loader,
        model_name='mobilenet_v2', train_val='Test', class_weights='None',
        time_taken=val_result['epoch_time']
    )
    
    # Append valing results
    val_results_list.append(val_result['metrics_df'])
    val_summary_list.append(val_summary)

    print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

    # Convert lists to DataFrames
    train_results_full_40 = pd.concat(train_results_list, ignore_index=True)
    train_results_full_40.to_csv("train_results_full_40.csv", index=False)
    
    train_summary_full_40 = pd.concat(train_summary_list, ignore_index=True)
    train_summary_full_40.to_csv("train_summary_full_40.csv", index=False)
    
    val_results_full_40 = pd.concat(val_results_list, ignore_index=True)
    val_results_full_40.to_csv("val_results_full_40.csv", index=False)
    
    val_summary_full_40 = pd.concat(val_summary_list, ignore_index=True)
    val_summary_full_40.to_csv("val_summary_full_40.csv", index=False)


🚀 Epoch 31 - Training Started...

Epoch 31, Batch 20, Loss: 0.0700, Accuracy: 98.12%, Time Passed: 0.69m
Epoch 31, Batch 40, Loss: 0.0565, Accuracy: 98.44%, Time Passed: 1.24m
Epoch 31, Batch 60, Loss: 0.0515, Accuracy: 98.49%, Time Passed: 1.79m
Epoch 31, Batch 80, Loss: 0.0538, Accuracy: 98.44%, Time Passed: 2.37m
Epoch 31, Batch 100, Loss: 0.0567, Accuracy: 98.41%, Time Passed: 2.95m
Epoch 31 Completed - Average Loss: 0.0565, Accuracy: 98.44%, Epoch Time: 3.33m
Training Complete Epoch 31 - Total Time: 3.33m

✅ Epoch 31 - Training Completed. Starting Evaluation...

Batch 20, Loss: 0.2972, Accuracy: 93.75%, Time Passed: 0.98m
Evaluation Complete - Loss: 0.2888, Accuracy: 94.07%, Total Time: 1.95m

📊 Epoch 31 - Evaluation Completed.


🚀 Epoch 32 - Training Started...

Epoch 32, Batch 20, Loss: 0.0460, Accuracy: 98.75%, Time Passed: 0.69m
Epoch 32, Batch 40, Loss: 0.0426, Accuracy: 98.75%, Time Passed: 1.30m
Epoch 32, Batch 60, Loss: 0.0420, Accuracy: 98.75%, Time Passed: 1.94m
Epoch 3

In [515]:
model_40 = model

In [516]:
models = {}
models['model_10'] = model_10
models['model_20'] = model_20
models['model_30'] = model_30
models['model_40'] = model_40

In [521]:
# Initialize lists to store data across epochs
train_results_list = []
train_summary_list = []
val_results_list = []
val_summary_list = []

for epoch in range(41, 51):
    print(f"\n🚀 Epoch {epoch} - Training Started...\n")
    
    # Training Phase
    training_result = train_model(model, train_loader, criterion, optimizer, train_dataset.classes, epoch_num=epoch, log_interval=20)
    
    train_summary = create_performance_summary(
        training_result['metrics_df'], training_result['overall_accuracy'], training_result['avg_loss'], train_loader,
        model_name='mobilenet_v2', train_val='Train', class_weights='None',
        time_taken=training_result['epoch_time']
    )

    
    # Append training results
    train_results_list.append(training_result['metrics_df'])
    train_summary_list.append(train_summary)

    print(f"\n✅ Epoch {epoch} - Training Completed. Starting Evaluation...\n")
    
    # Evaluation Phase
    val_result = evaluate_model(model, val_loader, criterion, val_dataset.classes, epoch_num=epoch, log_interval=20)

    val_summary = create_performance_summary(
        val_result['metrics_df'], val_result['overall_accuracy'], val_result['avg_loss'], val_loader,
        model_name='mobilenet_v2', train_val='Test', class_weights='None',
        time_taken=val_result['epoch_time']
    )

    models[f'model_{epoch}'] = model
    # Append valing results
    val_results_list.append(val_result['metrics_df'])
    val_summary_list.append(val_summary)

    print(f"\n📊 Epoch {epoch} - Evaluation Completed.\n")

    # Convert lists to DataFrames
    train_results_full_50 = pd.concat(train_results_list, ignore_index=True)
    train_results_full_50.to_csv("train_results_full_50.csv", index=False)
    
    train_summary_full_50 = pd.concat(train_summary_list, ignore_index=True)
    train_summary_full_50.to_csv("train_summary_full_50.csv", index=False)
    
    val_results_full_50 = pd.concat(val_results_list, ignore_index=True)
    val_results_full_50.to_csv("val_results_full_50.csv", index=False)
    
    val_summary_full_50 = pd.concat(val_summary_list, ignore_index=True)
    val_summary_full_50.to_csv("val_summary_full_50.csv", index=False)


🚀 Epoch 41 - Training Started...

Epoch 41, Batch 20, Loss: 0.0553, Accuracy: 98.12%, Time Passed: 0.65m
Epoch 41, Batch 40, Loss: 0.0536, Accuracy: 98.44%, Time Passed: 1.24m
Epoch 41, Batch 60, Loss: 0.0499, Accuracy: 98.33%, Time Passed: 1.82m
Epoch 41, Batch 80, Loss: 0.0457, Accuracy: 98.52%, Time Passed: 2.42m
Epoch 41, Batch 100, Loss: 0.0412, Accuracy: 98.66%, Time Passed: 3.06m
Epoch 41 Completed - Average Loss: 0.0439, Accuracy: 98.69%, Epoch Time: 3.48m
Training Complete Epoch 41 - Total Time: 3.48m

✅ Epoch 41 - Training Completed. Starting Evaluation...

Batch 20, Loss: 0.3193, Accuracy: 94.06%, Time Passed: 1.11m
Evaluation Complete - Loss: 0.4277, Accuracy: 91.71%, Total Time: 2.13m

📊 Epoch 41 - Evaluation Completed.


🚀 Epoch 42 - Training Started...

Epoch 42, Batch 20, Loss: 0.0225, Accuracy: 99.06%, Time Passed: 0.77m
Epoch 42, Batch 40, Loss: 0.0235, Accuracy: 99.14%, Time Passed: 1.47m
Epoch 42, Batch 60, Loss: 0.0255, Accuracy: 99.06%, Time Passed: 2.13m
Epoch 4

In [523]:
joblib.dump(models, 'models.pkl')
joblib.dump(optimizer, 'optimizer.pkl')

['optimizer.pkl']

In [ ]:
loaded_models = joblib.load('models.pkl')
loaded_models.keys()

In [ ]:
loaded_optimizers = joblib.load('optimizers.pkl')
loaded_optimizers.keys()

# Testing Model Generalization on Web Images

In [536]:
def evaluate_models_on_directory(models_dict, data_dir, class_names, display_images=False, particular_model=None):
    """
    Evaluate multiple models on a directory of plant images organized by subfolders.

    Args:
        models_dict: dict {model_name: model}
        data_dir: str, path to directory with subdirs (each subdir = class, contains images)
        class_names: list of class names
        display_images: bool, whether to display predictions with matplotlib
    
    Returns:
        results_df: DataFrame with predictions per image
        metrics_df: DataFrame with per-class precision, recall, f1 for each model
    """
    # Transform (same as validation)
    transform = v2.Compose([
        v2.ToImage(),
        v2.Resize(256),
        v2.CenterCrop(224),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
    ])

    # Storage
    records = []

    # Iterate over models
    for model_name, model in models_dict.items():
        if particular_model:
            if model_name != particular_model:
                continue
        model.eval()
        device = next(model.parameters()).device

        all_y_true = []
        all_y_pred = []

        # Walk through dataset
        for class_folder in os.listdir(data_dir):
            class_path = os.path.join(data_dir, class_folder)
            if not os.path.isdir(class_path):
                continue

            for img_file in os.listdir(class_path):
                img_path = os.path.join(class_path, img_file)
                if not (img_file.lower().endswith(('.png', '.jpg', '.jpeg'))):
                    continue

                # Load and preprocess image
                image = Image.open(img_path).convert("RGB")
                input_tensor = transform(image).unsqueeze(0).to(device)

                # Predict
                with torch.no_grad():
                    output = model(input_tensor)
                probs = torch.nn.functional.softmax(output[0], dim=0)
                confidence, pred_idx = torch.max(probs, 0)

                predicted_class = class_names[pred_idx.item()]
                actual_class = class_folder  # folder name = true class
                confidence_value = confidence.item() * 100

                # Save results
                records.append({
                    "model_name": model_name,
                    "image_path": img_path,
                    "actual_plant": actual_class,
                    "predicted_plant": predicted_class,
                    "confidence": confidence_value
                })

                all_y_true.append(class_names.index(actual_class))
                all_y_pred.append(pred_idx.item())

                # Optional display
                if display_images:
                    plt.figure(figsize=(6, 5))
                    plt.imshow(np.array(image))
                    plt.title(f"Actual: {actual_class}\nPredicted: {predicted_class} "
                              f"({confidence_value:.2f}%)\nEpoch num: {model_name[-2:]}")
                    plt.axis("off")
                    plt.show()

        # Compute metrics per class
        unique_classes = sorted(set(all_y_true))
        precisions = precision_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
        recalls = recall_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)
        f1s = f1_score(all_y_true, all_y_pred, labels=unique_classes, average=None, zero_division=0)

        # Add per-class metrics to DataFrame
        metrics_data = []
        for i, class_idx in enumerate(unique_classes):
            metrics_data.append([
                model_name,
                class_names[class_idx],
                precisions[i],
                recalls[i],
                f1s[i]
            ])

        # Add macro averages
        metrics_data.append([
            model_name,
            "Average Performance (Macro)",
            precisions.mean(),
            recalls.mean(),
            f1s.mean()
        ])

        # Merge into metrics_df
        if 'metrics_df' not in locals():
            metrics_df = pd.DataFrame(metrics_data, columns=["model_name", "plant_species", "precision", "recall", "f1_score"])
        else:
            metrics_df = pd.concat([metrics_df, pd.DataFrame(metrics_data, columns=["model_name", "plant_species", "precision", "recall", "f1_score"])],
                                   ignore_index=True)

    # Build results_df
    results_df = pd.DataFrame(records)

    return results_df, metrics_df

In [533]:
test_results_df.to_csv("test_results_df.csv", index=False)
test_metrics_df.to_csv("test_metrics_df.csv", index=False)

In [549]:
test_dir = "/Users/drmorsy/Downloads/Wadi Degla/Internet Images"
model_44_test_results_df, model_44_test_metrics_df = evaluate_models_on_directory(models, test_dir, train_dataset.classes,
                                                                                  display_images=False, particular_model='model_44')

In [550]:
model_44_test_metrics_df

,model_name,plant_species,precision,recall,f1_score
0,model_44,Capparis spinosa L.,0.740741,0.645161,0.689655
1,model_44,Diplotaxis harra (Forssk.) Boiss.,0.689655,0.512821,0.588235
2,model_44,Iphiona mucronata (Forssk.) Asch. & Schweinf.,0.416667,0.500000,0.454545
3,model_44,Ochradenus baccatus Delile,0.493151,0.818182,0.615385
4,model_44,Peganum harmala L.,0.857143,0.206897,0.333333
5,model_44,Tamarix nilotica (Ehrenb.) Bunge,0.750000,0.810811,0.779221
6,model_44,Average Performance (Macro),0.657893,0.582312,0.576729


In [551]:
accuracy = sum(model_44_test_results_df['actual_plant']==model_44_test_results_df['predicted_plant'])/len(model_44_test_results_df)*100
print(f"Accuracy on {len(model_44_test_results_df)} internet images: {accuracy:.2f}%")

Accuracy on 200 internet images: 61.00%
